# ***GRAPH CONVOLUTIONAL CLASSIFIER***

This notebook implements the solution proposed in the written part: a GCN-based node classifier using the similarity matrix W and the available labels.

# ***DATA LOADING AND INSPECTION***

First to make results repeatable across runs and machines

In [ ]:
SEED = 42

import os, random
import numpy as np

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)

try:
    tf.keras.utils.set_random_seed(SEED)
except AttributeError:
    tf.random.set_seed(SEED)

Now I load the data according to the instructions

In [ ]:
!wget -q https://frasca.di.unimi.it/MLDNN/input_data.zip -O data.zip

import zipfile, os, time, math, random
import pickle as pk
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score, accuracy_score
import itertools

t0 = time.time()

with zipfile.ZipFile("data.zip", 'r') as zip_ref:
    zip_ref.extractall("unzipped_data")

with open("unzipped_data/input_data.pkl", "rb") as f:
    data_dict = pk.load(f)

W = data_dict['similarity_matrix'].astype(np.float32)   # (5000, 5000), symmetric in [0,1]
labels = data_dict['labels'].astype(np.int64)           # (5000,), values in {-1, 0, 1}
true_labels = data_dict['true_labels'].astype(np.int64) # (5000,), ground truth
N = W.shape[0]

N, W.shape, labels.shape, true_labels.shape

(5000, (5000, 5000), (5000,), (5000,))

Let me print out the similarity matrix to see, if the preprocessing is needed.

In [ ]:
W

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.14142135, 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.14142135, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], dtype=float32)

# ***PREPROCESSING***

As I wrote in my exam sheet, I only add self-loops to the similarity matrix W. In practice, **I add ones on the diagonal where needed** so that **each node is connected to itself with weight 1**. Since W is symmetric, nothing more is needed. This is the standard setup for GCNs and is consistent with the propagation rule.
Self-loops ensure that a node keeps part of its own representation during message passing.

In [ ]:
np.fill_diagonal(W, 1.0)
W

array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.14142135, 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.14142135, ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]], dtype=float32)

# ***MODEL SETUP***



I will follow the scheme that I wrote about in my exam paper. Pre-processing FFNN → GCN layers → post-processing FFNN → classification head.

This cell builds the graph inputs from the similarity matrix W. It takes all non-zero entries of W as directed edges j→i, collects their weights W[i,j]

In [ ]:
rows, cols = np.nonzero(W)

edge_weights = W[rows, cols].astype(np.float32)
edges = np.vstack([rows, cols]).astype(np.int32)

edges_tf = tf.constant(edges, dtype=tf.int32)
edge_weights_tf = tf.constant(edge_weights, dtype=tf.float32)

num_nodes = W.shape[0]

I use this MLP, exactly gow we did in one of the labs

In [ ]:
def MLP(hidden_units, dropout_rate, kernel_regularizer=None, name=None):
    seq = []
    for units in hidden_units:
        seq.append(layers.BatchNormalization())
        seq.append(layers.Dropout(dropout_rate))
        seq.append(layers.Dense(units, activation=tf.nn.relu,
                                kernel_regularizer=kernel_regularizer))
    return keras.Sequential(seq, name=name)

Here I build the core of the project - **Graph Convolutional Layer**. I changed the implementation from the one we used in labs just for the speed, but IT IS NOT a change of my exam answer. It is still the GCN layer I claimed I'll use, just with a small optimization.

Instead of running the small MLP (“prepare”) E times (for every edge), I run it once for each node to produce prepared_nodes and then just gather those vectors along the edge list. Since the MLP is shared and depends only on the node's features (clarification about this a little bit lower), this is mathematically equivalent to the lab version.

The propagation rule and the outputs are the same; the change only reduces redundant MLP calls. On our sparse graph (~5k nodes, many edges) this cuts runtime substantially and helps respect the 5-minute constraint

In [ ]:
class GraphConvLayer(layers.Layer):
    """
    GCN layer similar to the one we used in Labs with message passing j -> i.
    Aggregation: "sum", "mean" (weighted or unweighted), or "max".
    """
    def __init__(self,
                 hidden_units,
                 dropout_rate=0.2,
                 aggregation_type="mean",
                 normalize=True,
                 kernel_regularizer=None,
                 name=None):
        super().__init__(name=name)
        self.aggregation_type = aggregation_type
        self.normalize = normalize

        self.prepare_fun = MLP(hidden_units, dropout_rate,
                               kernel_regularizer=kernel_regularizer, name="prepare")
        self.update_fun  = MLP(hidden_units, dropout_rate,
                               kernel_regularizer=kernel_regularizer, name="update")

    def call(self, inputs):
        node_repr, edges, edge_weights = inputs
        i = edges[0]
        j = edges[1]
        N = tf.shape(node_repr)[0]
        dtype = node_repr.dtype

        #Prepare ONCE per node, then gather by edges
        prepared_nodes = self.prepare_fun(node_repr)
        messages = tf.gather(prepared_nodes, j)

        if edge_weights is not None:
            messages = messages * tf.expand_dims(edge_weights, -1)

        #Aggregate to targets
        if self.aggregation_type == "sum":
            aggregated = tf.math.unsorted_segment_sum(messages, i, N)

        elif self.aggregation_type == "mean":
            aggregated = tf.math.unsorted_segment_sum(messages, i, N)
            if self.normalize:
                # degree: weighted sum of edge weights if provided, else neighbor count
                if edge_weights is not None:
                    deg = tf.math.unsorted_segment_sum(edge_weights, i, N)
                else:
                    ones = tf.ones_like(i, dtype=messages.dtype)
                    deg = tf.math.unsorted_segment_sum(ones, i, N)
                deg = tf.maximum(deg, tf.constant(1e-8, dtype=deg.dtype))
                aggregated = aggregated / tf.expand_dims(deg, -1)

        elif self.aggregation_type == "max":
            aggregated = tf.math.unsorted_segment_max(messages, i, N)

        else:
            raise ValueError(f"Unknown aggregation_type: {self.aggregation_type}")

        #Update node states
        out = self.update_fun(aggregated)
        return out

Now that I have the graph convolution layer, I can finally implement the classifier itself.

**CHANGE (more like clarification):** In the written solution I stated that I would apply a short FFNN preprocessing to the node features before the GCN layers. But we obviously have no features provided with the nodes in the dataset, so I will explain what I meant and what I am doing now. I treat each node **as having an identity (one-hot) feature vector and implement its linear projection using a learnable embedding matrix**. This is the same as applying a linear layer to one-hot features and fully matches the original pipeline (FFNN → GCN → FFNN → head) without introducing any extra information.

In [ ]:
class GCNNodeClassifier(tf.keras.Model):
    """
    Node classifier with the pipeline:
      short FFNN preprocessing (featureless) -> GCN x L -> short FFNN -> sigmoid head.

    Tunable hyperparameters:
      - num_layers: number of GCN layers
      - dropout_rate: dropout used inside MLP blocks
      - weight_decay: L2 regularization on Dense layers
    """
    def __init__(self,
                 num_nodes: int,
                 edges,
                 edge_weights,
                 hidden: int = 64,
                 num_layers: int = 2,
                 dropout_rate: float = 0.5,
                 weight_decay: float = 5e-4,
                 name: str = "GCNNodeClassifier"):
        super().__init__(name=name)

        self.num_nodes = int(num_nodes)
        self.edges = tf.constant(edges, dtype=tf.int32)
        self.edge_weights = tf.constant(edge_weights, dtype=tf.float32)

        self.reg = tf.keras.regularizers.l2(weight_decay) if weight_decay and weight_decay > 0 else None

        # --- short FFNN preprocessing ---
        # Treat node IDs as one-hot features and project them via a learnable embedding,then pass through a tiny FFNN (Linear + ReLU).
        self.embed = layers.Embedding(self.num_nodes, hidden, name="node_embedding")
        self.pre_ff = MLP([hidden], dropout_rate, kernel_regularizer=self.reg, name="pre_ff")

        # --- GCN stack (repeat L times) ---
        self.gconvs = [
            GraphConvLayer([hidden], dropout_rate,
                           aggregation_type="mean",
                           normalize=True,
                           kernel_regularizer=self.reg,
                           name=f"gconv_{l+1}")
            for l in range(num_layers)
        ]

        # --- short post-FFNN and final head ---
        self.post_ff = MLP([hidden], dropout_rate, kernel_regularizer=self.reg, name="post_ff")
        # Final head with sigmoid, as in the written plan
        self.head = layers.Dense(1, activation="sigmoid",
                                 kernel_regularizer=self.reg, name="head")

    def call(self, training=False):
        """
        Returns: tensor with probabilities.
        """
        # node IDs 0..N-1 -> embeddings -> short FFNN
        node_ids = tf.range(self.num_nodes, dtype=tf.int32)
        H = self.embed(node_ids)
        H = self.pre_ff(H, training=training)

        # Apply GCN layers
        for g in self.gconvs:
            H = g((H, self.edges, self.edge_weights))

        # Post-FFNN and sigmoid head
        H = self.post_ff(H, training=training)
        probs = tf.squeeze(self.head(H, training=training), axis=-1)
        return probs

    # Function for later evaluation
    def predict_classes(self, threshold=0.5, training=False):
        probs = self.call(training=training)
        return tf.cast(probs >= threshold, tf.int32)

# ***Hyperparameters tuning***

I split the known-label nodes into **train/validation (90/10)**.
For each configuration in a **small grid over #GCN layers L, dropout, and weight decay**, I train the model for a few epochs with early stopping and pick the configuration with the best validation Macro-F1.

In [ ]:
# Known/unknown masks and targets for training
labels_np = labels.copy()
known_mask = (labels_np != 0)
unknown_mask = (labels_np == 0)

# Map targets only technically for the loss: {-1, +1} -> {0,1}
y01 = (labels_np == 1).astype(np.float32)

# Train/val split on known nodes
rng = np.random.default_rng(SEED)
known_idx = np.where(known_mask)[0]
rng.shuffle(known_idx)
val_ratio = 0.10
val_size = max(1, int(len(known_idx) * val_ratio))
val_idx = known_idx[:val_size]
train_idx = known_idx[val_size:]

# Tensors for fast gather
y01_tf = tf.constant(y01, dtype=tf.float32)
train_idx_tf = tf.constant(train_idx, dtype=tf.int32)
val_idx_tf   = tf.constant(val_idx,   dtype=tf.int32)

The model's head outputs probabilities (sigmoid in the head), so I use BCE as a loss function

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)

def train_and_validate(L, dropout, wd, hidden=64, lr=1e-2, epochs=30, verbose=False):
    """Train a model with given hyperparams on train_idx; return val metrics + model."""
    model = GCNNodeClassifier(num_nodes=num_nodes,
                              edges=edges_tf,
                              edge_weights=edge_weights_tf,
                              hidden=hidden,
                              num_layers=L,
                              dropout_rate=dropout,
                              weight_decay=wd)
    opt = tf.keras.optimizers.Adam(learning_rate=lr)

    for ep in range(1, epochs+1):
        with tf.GradientTape() as tape:
            probs = model(training=True)
            y_pred = tf.gather(probs, train_idx_tf)
            y_true = tf.gather(y01_tf, train_idx_tf)
            loss = bce(y_true, y_pred)
            if model.losses:
                loss += tf.add_n(model.losses)
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients([(g, v) for g, v in zip(grads, model.trainable_variables) if g is not None])

        if verbose and (ep % 10 == 0 or ep == 1):
            print(f"[L={L} d={dropout} wd={wd}] ep={ep:02d} loss={float(loss):.4f}")

    # Validation metrics
    probs_val = model(training=False).numpy()
    yv_true = y01[val_idx]
    yv_pred = (probs_val[val_idx] >= 0.5).astype(int)
    val_acc = accuracy_score(yv_true, yv_pred) if yv_true.size else 0.0
    val_f1  = f1_score(yv_true, yv_pred, average='macro') if len(np.unique(yv_true)) > 1 else 0.0
    return model, val_acc, val_f1

This cell runs a tiny hyperparameter search to pick a good configuration quickly. It reports validation Macro-F1 and Accuracy. The best setup (by validation Macro-F1) is selected and its trained model is saved as best_model.

I use small grid to be under 5 minutes, but maybe with the bigger grid the result can be improved

In [ ]:
# I use small grid to stay well under 5 minutes.
grid_L  = [1, 2]          # number of GCN layers
grid_dr = [0.3, 0.5]      # dropout
grid_wd = [5e-4, 1e-3]    # weight decay

results = []
best = None

for L, dr, wd in itertools.product(grid_L, grid_dr, grid_wd):
    print(f"Config: L={L}, dropout={dr}, wd={wd}")
    model, v_acc, v_f1 = train_and_validate(L, dr, wd, hidden=32, lr=1e-2, epochs=20, verbose=False)
    print(f" -> val_f1={v_f1:.4f}  val_acc={v_acc:.4f}")
    rec = {"L": L, "dropout": dr, "wd": wd, "val_f1": v_f1, "val_acc": v_acc, "model": model}
    results.append(rec)
    if best is None or v_f1 > best["val_f1"]:
        best = rec

print("Best config:", {k: best[k] for k in ["L","dropout","wd","val_f1","val_acc"]})
best_model = best["model"]

Config: L=1, dropout=0.3, wd=0.0005
 -> val_f1=0.6350  val_acc=0.6941
Config: L=1, dropout=0.3, wd=0.001
 -> val_f1=0.3590  val_acc=0.5600
Config: L=1, dropout=0.5, wd=0.0005
 -> val_f1=0.3590  val_acc=0.5600
Config: L=1, dropout=0.5, wd=0.001
 -> val_f1=0.3590  val_acc=0.5600
Config: L=2, dropout=0.3, wd=0.0005
 -> val_f1=0.3590  val_acc=0.5600
Config: L=2, dropout=0.3, wd=0.001
 -> val_f1=0.3056  val_acc=0.4400
Config: L=2, dropout=0.5, wd=0.0005
 -> val_f1=0.3590  val_acc=0.5600
Config: L=2, dropout=0.5, wd=0.001
 -> val_f1=0.3590  val_acc=0.5600
Best config: {'L': 1, 'dropout': 0.3, 'wd': 0.0005, 'val_f1': 0.6350340855044126, 'val_acc': 0.6941176470588235}


# ***FINAL EVALUATION***

In [ ]:
# Final evaluation on the originally unknown nodes
unknown_idx = np.where(unknown_mask)[0]
if unknown_idx.size > 0:
    probs_all = best_model(training=False).numpy()
    y_true_unknown = (true_labels[unknown_idx] == 1).astype(int)
    y_pred_unknown = (probs_all[unknown_idx] >= 0.5).astype(int)
    acc_unknown = accuracy_score(y_true_unknown, y_pred_unknown)
    f1_unknown  = f1_score(y_true_unknown, y_pred_unknown, average='macro')
    print(f"Unknown-set Accuracy: {acc_unknown:.4f}")
    print(f"Unknown-set Macro-F1: {f1_unknown:.4f}")
else:
    print("No unknown nodes to evaluate.")

Unknown-set Accuracy: 0.6973
Unknown-set Macro-F1: 0.6471
